In [7]:
"""
================================================================================
SCRIPT NAME: Table 1 - Regional and Global Landward Migration (m) & SLR (1985-2025)
             (Aligned with Figure 2 filters, percentiles, and 1985-2025 period;
              excludes S19E063 due to erroneous water baseline raster)

AUTHOR: R. Walcker, CRBE
================================================================================
"""

from pathlib import Path
import warnings
import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.stats import linregress

warnings.filterwarnings("ignore")

# ==============================================================================
# 0. CONFIGURATION & PATHS
# ==============================================================================
OUT_DIR = Path(r"D:\MANGROVES\Research_Article_Clean\Table1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARLY_STATS_DIR = Path(r"D:\MANGROVES\Research_Article_Clean\Created_Data\Distance_CSVs")
SHP_VAL_PATH = Path(r"D:\MANGROVES\Research_Article_Clean\Created_Data\gmwv4_mask_anthropogenic_validation.shp")
CSV_MSL_PATH = Path(r"D:\MANGROVES\Research_Article_Clean\Created_Data\ecmwf_msl_annual_1deg_tiles.csv")
CSV_MANGROVES = Path(r"D:\MANGROVES\Research_Article_Clean\Created_Data\gmwv4_surfaces_annual_1deg_tiles.csv")

MAX_DIST_LIMIT = 20000
START_YEAR = 1985
END_YEAR = 2025

# Manual exclusions (e.g., erroneous water baseline raster)
EXCLUDED_TILES = {"S19E063"}

MIG_COLS = [
    "tile_id",
    "center_lon",
    "center_lat",
    "cumul_shift_1985_2025_m",
]

# ==============================================================================
# 1. STUDY REGIONS DEFINITION (MATCHING FIGURE 2)
# ==============================================================================
study_regions = [
    {"title": "Caribbean", "lat_range": (10.0, 30.0), "lon_range": (-90.0, -60.0)},
    {"title": "West Africa", "lat_range": (10.0, 25.0), "lon_range": (-20.0, -10.0)},
    {"title": "Middle East", "lat_range": (10.0, 30.0), "lon_range": (25.0, 70.0)},
    {"title": "Northern Australia", "lat_range": (-20.0, -10.0), "lon_range": (125.0, 150.0)},
    {"title": "New Zealand", "lat_range": (-40.0, -30.0), "lon_range": (170.0, 180.0)},
]

# ==============================================================================
# 2. DATA LOADING & ISOLATING NATURAL TILES
# ==============================================================================
print("⏳ Loading spatial grid and isolating natural tiles...")

gdf_val = gpd.read_file(SHP_VAL_PATH)
tile_col_val = "clean_id" if "clean_id" in gdf_val.columns else "tile_id"
gdf_val["tile_id_clean"] = gdf_val[tile_col_val].astype(str).str.strip().str.upper()

if gdf_val.crs is not None and gdf_val.crs.to_string() != "EPSG:4326":
    gdf_val = gdf_val.to_crs(epsg=4326)

manual_mask_dict = dict(zip(gdf_val["tile_id_clean"], gdf_val["mask_artif"]))
excluded_upper = {t.strip().upper() for t in EXCLUDED_TILES}
nat_tiles_set = {
    tid for tid, artif in manual_mask_dict.items()
    if artif == 0 and tid not in excluded_upper
}

df_mng_raw = pd.read_csv(CSV_MANGROVES, sep=";")
if len(df_mng_raw.columns) == 1:
    df_mng_raw = pd.read_csv(CSV_MANGROVES, sep=",")
df_mng_raw["tile_id"] = df_mng_raw["tile_id"].astype(str).str.strip().str.upper()
df_mng_raw["year"] = df_mng_raw["year"].astype(int)

df_msl_raw = pd.read_csv(CSV_MSL_PATH, sep=";")
if len(df_msl_raw.columns) == 1:
    df_msl_raw = pd.read_csv(CSV_MSL_PATH, sep=",")
df_msl_raw["tile_id"] = df_msl_raw["tile_id"].astype(str).str.strip().str.upper()
df_msl_raw["year"] = df_msl_raw["year"].astype(int)

coords_df = (
    df_msl_raw[["tile_id", "center_lon", "center_lat"]]
    .drop_duplicates(subset=["tile_id"])
    .set_index("tile_id")
)
coords_dict = coords_df.to_dict(orient="index")

df_msl_data = df_msl_raw.pivot(index="year", columns="tile_id", values="SSH_m")
df_msl_data.index = df_msl_data.index.astype(int)
df_msl_data.sort_index(ascending=True, inplace=True)
df_msl_data.columns = df_msl_data.columns.astype(str).str.strip().str.upper()

df_mng_nat = df_mng_raw[df_mng_raw["tile_id"].isin(nat_tiles_set)].copy()
pivot_mng = df_mng_nat.pivot(index="year", columns="tile_id", values="area_km2")

common_nat_tiles = pivot_mng.columns.intersection(df_msl_data.columns)
pivot_mng = pivot_mng[common_nat_tiles]
df_msl_data = df_msl_data[common_nat_tiles]

# ==============================================================================
# 3. PER-TILE FILTERING (cumul_shift 1985-2025 > 0) IN METERS
# ==============================================================================
print(f"⏳ Filtering natural tiles (excluding {sorted(excluded_upper)}) with positive 1985-2025 landward migration (m)...")
if not YEARLY_STATS_DIR.exists():
    print(f"⚠️ Directory not found: {YEARLY_STATS_DIR}")

tile_records = []
for tile_id in sorted(nat_tiles_set):
    if tile_id in excluded_upper or tile_id not in coords_dict:
        continue

    csv_path = YEARLY_STATS_DIR / f"mangrove_distances_yearly_{tile_id}.csv"
    if not csv_path.exists():
        continue

    try:
        df_z = pd.read_csv(csv_path, sep=";")
        if len(df_z.columns) == 1:
            df_z = pd.read_csv(csv_path, sep=",")
        if df_z.empty or "year" not in df_z.columns or "Dist_to_sea" not in df_z.columns:
            continue

        df_z["year"] = pd.to_numeric(df_z["year"], errors="coerce")
        df_z["Dist_to_sea"] = pd.to_numeric(df_z["Dist_to_sea"], errors="coerce")
        df_z = df_z.dropna(subset=["year", "Dist_to_sea"]).copy()
        df_z["year"] = df_z["year"].astype(int)

        yearly_maxs = {}
        for yr, grp in df_z.groupby("year"):
            dvals = grp["Dist_to_sea"].to_numpy(dtype=float)
            dvals_filtered = dvals[(dvals >= 0) & (dvals <= MAX_DIST_LIMIT)]
            if len(dvals_filtered) > 0:
                yearly_maxs[int(yr)] = float(np.max(dvals_filtered))

        # Match Figure 2 filter: require 1985 and 2025 with positive cumulative shift > 0
        if START_YEAR not in yearly_maxs or END_YEAR not in yearly_maxs:
            continue

        cumul_shift_85_25 = yearly_maxs[END_YEAR] - yearly_maxs[START_YEAR]
        if cumul_shift_85_25 <= 0:
            continue

        tile_records.append({
            "tile_id": tile_id,
            "center_lon": float(coords_dict[tile_id]["center_lon"]),
            "center_lat": float(coords_dict[tile_id]["center_lat"]),
            "cumul_shift_1985_2025_m": cumul_shift_85_25,
        })
    except Exception:
        continue

df_tiles_mig = pd.DataFrame(tile_records, columns=MIG_COLS)

# ==============================================================================
# 4. REGIONAL & GLOBAL AGGREGATION FOR TABLE 2 (1985-2025, IN METERS)
# ==============================================================================
def _summarize_migration_meters(vals: pd.Series) -> dict[str, object]:
    """Summarize cumulative landward migration (m) over 1985-2025 using Figure 2 conventions."""
    clean = pd.to_numeric(vals, errors="coerce").dropna()
    if clean.empty:
        return {
            "Landward migration Mean ± SD (m)": np.nan,
            "Landward migration Q25–Q75 (m)": np.nan,
            "Landward migration Max (m)": np.nan,
        }

    mean_val = float(clean.mean())
    sd_val = float(clean.std(ddof=1)) if len(clean) > 1 else 0.0
    q25_val = float(clean.quantile(0.25))
    q75_val = float(clean.quantile(0.75))
    max_val = float(clean.max())

    return {
        "Landward migration Mean ± SD (m)": f"{mean_val:.0f} ± {sd_val:.0f}",
        "Landward migration Q25–Q75 (m)": f"{q25_val:.0f}–{q75_val:.0f}",
        "Landward migration Max (m)": round(max_val, 0),
    }


def _compute_slr_mean_sd(tile_ids: list[str], start_year: int, end_year: int) -> str | float:
    """Compute SLR trend Mean ± SD (mm/yr) across tiles over [start_year, end_year]."""
    if df_msl_data.empty or not tile_ids:
        return np.nan

    sub_years = [y for y in df_msl_data.index if start_year <= int(y) <= end_year]
    if len(sub_years) < 2:
        return np.nan

    x_yrs = np.asarray(sub_years, dtype=float)
    rates = []

    for t_id in tile_ids:
        if t_id in excluded_upper or t_id not in df_msl_data.columns:
            continue
        # Convert sea surface height from meters to millimeters before linear trend fit
        y_mm = df_msl_data.loc[sub_years, t_id].to_numpy(dtype=float) * 1000.0
        valid = np.isfinite(x_yrs) & np.isfinite(y_mm)
        if np.sum(valid) >= 2 and np.std(x_yrs[valid]) > 0:
            slope, _, _, _, _ = linregress(x_yrs[valid], y_mm[valid])
            if np.isfinite(slope):
                rates.append(float(slope))

    if not rates:
        return np.nan

    arr = np.asarray(rates, dtype=float)
    mean_val = float(np.mean(arr))
    sd_val = float(np.std(arr, ddof=1)) if arr.size > 1 else 0.0
    return f"{mean_val:.2f} ± {sd_val:.2f}"


def _build_region_row(region_name: str, sub_df: pd.DataFrame) -> dict[str, object]:
    """Assemble one Table 1 row aligned with Figure 2."""
    tile_list = [
        t for t in (sub_df["tile_id"].astype(str).str.strip().str.upper().tolist() if not sub_df.empty else [])
        if t not in excluded_upper
    ]
    row: dict[str, object] = {"Region": region_name}
    row.update(_summarize_migration_meters(sub_df["cumul_shift_1985_2025_m"]))
    row["Sea-level rise Mean ± SD (mm/yr)"] = _compute_slr_mean_sd(tile_list, START_YEAR, END_YEAR)
    return row


regional_rows = []
for reg in study_regions:
    lat_min, lat_max = reg["lat_range"]
    lon_min, lon_max = reg["lon_range"]

    if df_tiles_mig.empty:
        sub = df_tiles_mig
    else:
        mask = (
            (df_tiles_mig["center_lat"] >= lat_min)
            & (df_tiles_mig["center_lat"] <= lat_max)
            & (df_tiles_mig["center_lon"] >= lon_min)
            & (df_tiles_mig["center_lon"] <= lon_max)
        )
        sub = df_tiles_mig[mask]

    regional_rows.append(_build_region_row(reg["title"], sub))

# Append Global summary row at the end of the table
global_row = _build_region_row("Global", df_tiles_mig)
df_table2 = pd.DataFrame(regional_rows + [global_row])

# ==============================================================================
# 5. DISPLAY & EXPORT (CSV + XLSX)
# ==============================================================================
print("\n" + "=" * 130)
print("TABLE 1: REGIONAL AND GLOBAL LANDWARD MIGRATION (m) & SLR (mm/yr) — 1985-2025 (excl. S19E063)")
print("=" * 130)
print(df_table2.iloc[:-1].to_string(index=False))
print("-" * 130)
print(df_table2.iloc[-1:].to_string(index=False, header=False))
print("=" * 130)

csv_out_path = OUT_DIR / "Table1.csv"
xlsx_out_path = OUT_DIR / "Table1.xlsx"

df_table2.to_csv(csv_out_path, sep=";", index=False)
try:
    df_table2.to_excel(xlsx_out_path, index=False)
    print(f"✅ Table 1 saved to:\n  - {csv_out_path}\n  - {xlsx_out_path}")
except Exception as exc:
    print(f"✅ Table 1 saved to CSV: {csv_out_path} (Excel export skipped: {exc})")

⏳ Loading spatial grid and isolating natural tiles...
⏳ Filtering natural tiles (excluding ['S19E063']) with positive 1985-2025 landward migration (m)...

TABLE 2: REGIONAL AND GLOBAL LANDWARD MIGRATION (m) & SLR (mm/yr) — 1985-2025 (excl. S19E063)
            Region Landward migration Mean ± SD (m) Landward migration Q25–Q75 (m)  Landward migration Max (m) Sea-level rise Mean ± SD (mm/yr)
         Caribbean                        108 ± 160                         17–134                       877.0                      3.55 ± 1.14
       West Africa                        769 ± 662                        286–981                      1824.0                      3.05 ± 0.11
       Middle East                        147 ± 192                         28–193                       965.0                      3.14 ± 0.27
Northern Australia                        335 ± 593                         36–300                      2972.0                      3.67 ± 0.19
       New Zealand             